# Anatomy of a RAG Pipeline — companion notebook

Bisect a wrong answer to the stage that caused it, and see why a score threshold cannot fix a *truncated* chunk.

## 1. The pipeline

Sentence chunking, keyword retrieval, a generator that answers only from what it is given.

In [ ]:
SOURCE = ("Sale and clearance items may be returned unused within 30 days "
          "only if the item is faulty. Non-faulty sale items are final.")
QUESTION, TRUTH = "Can I return a sale item?", "no"
STOP = {"can","i","a","the","are","is","to","may","be","and","only","if","within"}

def chunk(t): return [s.strip() + "." for s in t.split(".") if s.strip()]

def keywords(s):
    return {w[:4] for w in s.replace("?"," ").replace("."," ").lower().split()
            if len(w) > 2 and w not in STOP}

def retrieve(chunks, q, k, threshold):
    qk = keywords(q)
    sc = sorted(((len(qk & keywords(c))/max(1,len(qk)), c) for c in chunks),
                reverse=True, key=lambda t: t[0])
    return [(s,c) for s,c in sc[:k] if s >= threshold], sc[:k]

def generate(ctx):
    j = " ".join(ctx).lower()
    if "final" in j: return "no"
    if "may be returned" in j: return "yes"
    return "unknown"

for i, c in enumerate(chunk(SOURCE), 1):
    print(f"chunk {i}: {c}")

## 2. Sweep k and the threshold together

The grid below shows every combination. Look for the pattern: **the threshold column does nothing**, because the misleading chunk scores a perfect 1.00.

In [ ]:
print(f"{'k':>3} {'thresh':>7} {'answer':>9} {'verdict':>9}")
grid = {}
for k in (1, 2, 3):
    for th in (0.0, 0.3, 0.6, 0.9):
        kept, _ = retrieve(chunk(SOURCE), QUESTION, k, th)
        ans = generate([c for _s, c in kept])
        grid[(k, th)] = ans
        ok = "CORRECT" if ans == TRUTH else "wrong"
        print(f"{k:>3} {th:>7.1f} {ans:>9} {ok:>9}")

print("\nk=1 is wrong at EVERY threshold. k>=2 is correct at every threshold")
print("below 0.9. The fix is retrieving the neighbour, not filtering by score.")

## 3. Why: look at the scores

The chunk that *decides* the answer scores **lower** than the one that misleads. Similarity is not importance.

In [ ]:
_kept, top = retrieve(chunk(SOURCE), QUESTION, 3, 0.0)
for s, c in top:
    decides = "  <- DECIDES the answer" if "final" in c.lower() else ""
    print(f"  {s:.2f}  {c}{decides}")
print("\nA threshold removes LOW-scoring chunks. The problem here is a")
print("HIGH-scoring chunk that is incomplete. Different failure, different fix.")

## 4. The four retrieval failures

Only one of them is a threshold problem. Worth memorising.

In [ ]:
rows = [
    ("irrelevant",  "nothing in the corpus answers this", "LOW  (0.2)", "threshold -> abstain"),
    ("truncated",   "right passage, cut short",           "HIGH (1.0)", "chunking / overlap / bigger k"),
    ("confusable",  "same topic, wrong specifics",        "HIGH (0.95)", "hybrid or exact matching"),
    ("stale",       "right passage, obsolete",            "HIGH",        "corpus hygiene, date filters"),
]
print(f"{'failure':<12} {'what happened':<38} {'score':<12} fix")
for r in rows:
    print(f"{r[0]:<12} {r[1]:<38} {r[2]:<12} {r[3]}")
print("\nAdding a threshold and calling retrieval 'handled' fixes 1 of 4.")

## 5. Your turn

1. Replace `SOURCE` with a real policy of yours that has an exception clause.
2. Find a question where `k=1` gives the opposite of the truth.
3. Log the retrieved chunks next to every answer in your own system — that logging is what turns a guess into a bisection.